## Hadoop: Writing and Reading Data with HDFS

This notebook demonstrates how to interact with HDFS (Hadoop Distributed File System) using Python through the **HttpFS proxy** (port 14000) — no `subprocess` or `docker exec` needed. We cover the fundamental operations of writing and reading data, and then peek behind the curtain to see how HDFS **replicates** the file across the cluster.

#### 1. Connect to HDFS via the proxy (HttpFS)

Creates a client pointing to the proxy (HttpFS, port 14000), which acts as a proxy
and resolves DataNode hostnames internally — no subprocess needed for any operation.


In [ ]:
from hdfs import InsecureClient

client = InsecureClient('http://localhost:14000', user='root')
print('Connected to proxy!')

#### 2. Explore HDFS

Lists files and folders in the HDFS root. This operation uses the NameNode directly (no redirect), so it works fine.

In [ ]:
files = client.list('/')
print(f'Files in HDFS root: {files}')

#### 3. Download sample data

Downloads the Titanic dataset directly from GitHub to the `temp/` directory.

In [ ]:
import sys; sys.path.insert(0, '../scripts')
import os
from hdfs_utils import download_stream

os.makedirs('../temp', exist_ok=True)
download_stream('https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv', '../temp/sample_data.csv')

#### 4. Upload to HDFS via the proxy

Using the proxy (HttpFS), uploads go through a proxy that resolves DataNode hostnames
internally. The `hdfs` library communicates with HDFS as if it were a single endpoint.


In [ ]:
# Upload via proxy (handles DataNode routing internally)
client.upload(
    hdfs_path='/user/root/data.csv',
    local_path='../temp/sample_data.csv',
    overwrite=True,
    permission=775,
)
print('File uploaded successfully! ➔ /user/root/data.csv')

# Verify the file is there
files = client.list('/user/root/')
print(f'Files: {files}')

#### 5. Read data from HDFS

Read the uploaded file directly into a Pandas DataFrame via the proxy.


In [ ]:
import pandas as pd
import io

# Read the file from HDFS via the proxy and load into Pandas
with client.read('/user/root/data.csv') as reader:
    df = pd.read_csv(io.StringIO(reader.read().decode('utf-8')))
df

#### 6. Behind the scenes: blocks & replication

Writing a file felt like writing to a single disk — but HDFS quietly **split it into blocks** and stored a **replica of every block on 3 different DataNodes** (the `dfs.replication = 3` we configured). Let's prove it, all from Python via the proxy.

Our Titanic file is tiny (~60 KB), well under the 128 MB block size, so it is a **single block** — but that block still lives on **3 machines** for fault tolerance. (Lab 02 shows a bigger file fragmenting into *several* blocks.)

In [ ]:
import sys; sys.path.insert(0, '../scripts')
from hdfs_utils import block_report

In [ ]:
# One small block, but replicated across 3 DataNodes:
block_report(client, '/user/root/data.csv')

Notice the **same block appears on 3 hosts** (`datanode1`, `datanode2`, `datanode3`). If any one DataNode dies, the NameNode still has 2 good copies and re-replicates the block automatically.

👉 See it visually: **[localhost:9870 → Utilities → Browse the file system](http://localhost:9870/explorer.html#/user/root)** → click `data.csv` → *Block information*.

#### 7. Cleanup (optional)

Removes the locally downloaded CSV file.

In [ ]:
import os

if os.path.exists('../temp/sample_data.csv'):
    os.remove('../temp/sample_data.csv')
    print('Local file removed.')